# Edge AI Assignment 1: Module 1 — How Long Will the Battery Last?
# Data Inspection: Team 3 Telemetry Data

* **Team:** 3 — **Phoenix**
* **Members:** Pavan R, Shreyank D K, Adarsh K R, Ankitha A
* **Dataset:** `data/team_03_train.csv`

---
> ### 💡 Core Principle
> **"Don't start cleaning data or training a model before understanding what the data is telling you."**
> 
> In this notebook, we follow a systematic, step-by-step Data Inspection workflow to inspect our raw telemetry records without prematurely modifying, normalizing, or splitting the dataset.


## Step 1 — Load the Dataset
First, load the raw CSV into a Pandas DataFrame. At this point, we do not modify anything; we just ingest the raw data.


In [1]:
# Step 1: Load raw dataset
import os
import pandas as pd
import numpy as np

csv_path = "../data/team_03_train.csv" if os.path.exists("../data/team_03_train.csv") else "data/team_03_train.csv"
df = pd.read_csv(csv_path)

print("Dataset loaded successfully into DataFrame 'df'.")


Dataset loaded successfully into DataFrame 'df'.


## Step 2 — Inspect the Basic Structure
We run:
1. `df.head()` → see how the dataset begins.
2. `df.tail()` → see how it ends.
3. `df.sample(5)` → avoid examining only the first few records; get a glimpse of randomly selected observations across the entire dataset.

**What does a single row represent?**
One observation from a smart doorbell device, containing 5 telemetry measurements and the target remaining battery life (`days_left`).


In [2]:
# Step 2: head(), tail(), and sample(5)
print("--- 1. First 5 Rows (head) ---")
display(df.head())

print("\n--- 2. Last 5 Rows (tail) ---")
display(df.tail())

print("\n--- 3. Random Sample of 5 Rows (sample) ---")
display(df.sample(5, random_state=42))


--- 1. First 5 Rows (head) ---


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
0,3.476,24.0,0,-64,26,7.2
1,3.934,17.6,7,-46,27,117.9
2,4.020,29.7,13,-45,21,106.0
3,3.431,34.4,4,-81,12,1.6
4,4.082,33.4,13,-60,11,105.8



--- 2. Last 5 Rows (tail) ---


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
1995,3.419,23.8,5,-73,27,1.6
1996,3.619,32.7,3,-48,35,26.2
1997,4.109,29.5,5,-68,9,167.8
1998,4.156,39.3,6,-70,35,74.1
1999,4.145,28.5,9,-84,21,49.5



--- 3. Random Sample of 5 Rows (sample) ---


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
1860,3.803,26.0,9,-46,23,87.8
353,3.796,31.0,7,-60,29,76.4
1333,3.488,26.7,2,-71,27,6.1
905,4.139,25.4,5,-47,6,223.3
1289,4.183,29.5,14,-68,23,70.7


## Step 3 — Understand the Dimensions of the Dataset
We check the shape of the dataset:
$$\text{shape} = (\text{number of rows}, \text{number of columns})$$


In [3]:
# Step 3: Dataset dimensions
shape = df.shape
print(f"df.shape: {shape}")
print(f"Number of observations (rows)   : {shape[0]}")
print(f"Number of total columns         : {shape[1]} (5 input features + 1 target variable)")


df.shape: (2000, 6)
Number of observations (rows)   : 2000
Number of total columns         : 6 (5 input features + 1 target variable)


## Step 4 — Inspect Column Names
We ensure:
* All expected columns are present.
* No unexpected or corrupted columns exist.
* Column names are correctly spelled.
* We know exactly which column is the target.

| Column | Meaning | Role | Physical Unit |
| :--- | :--- | :--- | :--- |
| `battery_v` | Battery terminal voltage | Feature | Volts ($V$) |
| `temp_c` | Ambient temperature | Feature | Celsius ($^\circ C$) |
| `rings_per_day` | Doorbell chime / activity count | Feature | Integer count |
| `wifi_rssi_dbm` | Wi-Fi signal strength | Feature | $dBm$ (closer to 0 is stronger) |
| `age_months` | Device / battery age | Feature | Months |
| `days_left` | Remaining battery life | **Target** | Days |


In [4]:
# Step 4: Verify columns
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    role = "TARGET" if col == "days_left" else "FEATURE"
    print(f"  {i}. {col:15s} [{role}]")


Column Names:
  1. battery_v       [FEATURE]
  2. temp_c          [FEATURE]
  3. rings_per_day   [FEATURE]
  4. wifi_rssi_dbm   [FEATURE]
  5. age_months      [FEATURE]
  6. days_left       [TARGET]


## Step 5 — Inspect Data Types
We run `df.info()` to verify data types, non-null counts, and memory usage.

**Why does this matter?**
If `battery_v` was accidentally loaded as a string (e.g., `"3.48"` instead of float `3.48`), an ML model cannot perform mathematical operations on it without type conversion.


In [5]:
# Step 5: Data types and memory info
print("df.info():")
df.info()


df.info():
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   battery_v      2000 non-null   float64
 1   temp_c         2000 non-null   float64
 2   rings_per_day  2000 non-null   int64  
 3   wifi_rssi_dbm  2000 non-null   int64  
 4   age_months     2000 non-null   int64  
 5   days_left      2000 non-null   float64
dtypes: float64(3), int64(3)
memory usage: 93.9 KB


## Task 1: Missing-Value Inspection

We check for missing (null / NaN) values across all columns using `df.isnull().sum()` and compute the percentage of missing values per column.


In [6]:
# Task 1: Missing values inspection
# 1. Display the number of missing values in each column
print("Number of missing values per column:")
missing_counts = df.isnull().sum()
print(missing_counts)

# 2. Display the percentage of missing values in each column
print("\nPercentage of missing values per column (%):")
missing_percentages = (df.isnull().sum() / len(df)) * 100
print(missing_percentages)

# Summary tabular view
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages
})
display(missing_summary)


Number of missing values per column:
battery_v        0
temp_c           0
rings_per_day    0
wifi_rssi_dbm    0
age_months       0
days_left        0
dtype: int64

Percentage of missing values per column (%):
battery_v        0.0
temp_c           0.0
rings_per_day    0.0
wifi_rssi_dbm    0.0
age_months       0.0
days_left        0.0
dtype: float64


,Missing Count,Percentage (%)
battery_v,0,0.0
temp_c,0,0.0
rings_per_day,0,0.0
wifi_rssi_dbm,0,0.0
age_months,0,0.0
days_left,0,0.0


### Interpretation of Missing Values:
* **Zero Missing Values:** Every column (`battery_v`, `temp_c`, `rings_per_day`, `wifi_rssi_dbm`, `age_months`, and `days_left`) contains exactly **0 missing values (0.00%)** out of all 2,000 rows.
* **Target Completeness:** The target variable `days_left` is completely populated, meaning all 2,000 rows have valid target labels.
* **No Imputation Required:** Because there are no missing / NaN entries, we do not need to apply any missing-value imputation (mean/median replacement) or deletion at this stage.
* *Domain Note:* In embedded sensor data, "missing" readings often do not manifest as NaN, but rather as hardware fault codes (e.g., `-127.0°C` or `0.0V`). We will investigate these hardware codes during range and glitch inspection.


## Task 2: Duplicate Inspection

We check for fully duplicated rows using `df.duplicated().sum()` and inspect any identified duplicate records.


In [7]:
# Task 2: Duplicate records inspection
# 1. Calculate the number of fully duplicated rows
dup_count = df.duplicated().sum()
print(f"Number of fully duplicated rows: {dup_count}")

# 2. Display duplicates if they exist (without modifying df)
print("\nDuplicate rows:")
display(df[df.duplicated()])


Number of fully duplicated rows: 0

Duplicate rows:


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left


### Interpretation of Duplicate Inspection:
* **Zero Duplicate Rows:** `df.duplicated().sum()` returns **0**. Every single record in the 2,000-row dataset has a unique combination of features and target value.
* **What Duplicates Mean:** An exact duplicate occurs when every feature value and the target variable match an existing observation across all columns.
* **Why Investigate Before Removing:** If duplicates were present, we would need to investigate whether they were caused by logging retry glitches (e.g., duplicate packet transmissions from network retry) or legitimate identical sensor states before deciding to remove them. Removing rows prematurely can distort natural frequency distributions.
* **No Deletions Made:** The original DataFrame `df` remains strictly untouched and preserved with all 2,000 records.


## Step 8 — Numerical Summary Statistics
We run `df.describe()` to inspect count, mean, std, min, 25%, 50% (median), 75%, and max.
This is one of the most critical inspection steps to spot physically impossible values.


In [8]:
# Step 8: Numerical summary statistics
print("Numerical Summary Statistics:")
display(df.describe().T)


Numerical Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
battery_v,2000.0,3.766676,0.351431,0.0,3.5715,3.7845,4.000,4.2
temp_c,2000.0,27.769550,10.313402,-127.0,23.5000,28.2500,32.800,46.0
rings_per_day,2000.0,7.857000,5.550221,0.0,4.0000,6.5000,11.000,38.0
wifi_rssi_dbm,2000.0,-65.758500,13.260358,-88.0,-77.0000,-65.0000,-55.000,0.0
age_months,2000.0,18.067000,10.788777,0.0,9.0000,18.0000,27.000,36.0
days_left,2000.0,65.798950,57.919885,0.1,12.5000,54.8000,104.575,317.1


## Step 9 — Inspect Each Feature's Range & Physical Sanity

We explicitly examine min, max, mean, median, and std, and ask key domain questions:

| Feature | Questions We Ask | Physical Sanity Check |
| :--- | :--- | :--- |
| `battery_v` | Are voltages physically plausible? | Operating range is $3.0\text{ V} - 4.25\text{ V}$. **$0.0\text{ V}$ is impossible for an active transmitting device!** |
| `temp_c` | Are temperatures realistic? | Outdoor range is $-20^\circ\text{C} - 50^\circ\text{C}$. **$-127^\circ\text{C}$ is a digital sensor error code!** |
| `rings_per_day` | Are activity counts reasonable? | Counts must be $\ge 0$. |
| `wifi_rssi_dbm` | Are signal values realistic? | Wi-Fi RSSI is negative ($-90$ to $-30\text{ dBm}$). **$0\text{ dBm}$ indicates unconnected / default state!** |
| `age_months` | Are device ages reasonable? | Age must be $\ge 0$. |
| `days_left` | Are target values valid? | Remaining life must be $\ge 0$. |


In [9]:
# Step 9: Detailed range and sanity audit table
range_data = []
for col in df.columns:
    q25 = df[col].quantile(0.25)
    median = df[col].median()
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25
    range_data.append({
        'Feature': col,
        'Min': round(df[col].min(), 3),
        'Max': round(df[col].max(), 3),
        'Mean': round(df[col].mean(), 3),
        'Median': round(median, 3),
        'Std Dev': round(df[col].std(), 3),
        'IQR': round(iqr, 3)
    })

display(pd.DataFrame(range_data))


,Feature,Min,Max,Mean,Median,Std Dev,IQR
0,battery_v,0.0,4.2,3.767,3.784,0.351,0.429
1,temp_c,-127.0,46.0,27.770,28.250,10.313,9.300
2,rings_per_day,0.0,38.0,7.857,6.500,5.550,7.000
3,wifi_rssi_dbm,-88.0,0.0,-65.758,-65.000,13.260,22.000
4,age_months,0.0,36.0,18.067,18.000,10.789,18.000
5,days_left,0.1,317.1,65.799,54.800,57.920,92.075


## Step 14 & 15 — Inspect Unique Values, Value Patterns & Suspicious Readings

We examine distinct value counts and frequency distributions for key columns to identify:
1. `rings_per_day` unique counts and distribution.
2. Suspicious values:
   - `battery_v == 0.0`
   - `temp_c == -127.0`
   - `wifi_rssi_dbm == 0`
   - Negative values in rings, age, or days_left.

**Decision Table:**
| Observation | Hardware Interpretation | Proposed Action |
| :--- | :--- | :--- |
| `battery_v == 0.0` | ADC dropout / power dip during packet transmit | Filter out from training; reject in API with 422 |
| `temp_c == -127.0` | 1-Wire DS18B20 digital disconnect error code | Filter out from training; reject in API with 422 |
| `wifi_rssi_dbm == 0` | Wi-Fi stack unassociated / disconnected default 0 | Filter out from training; reject in API with 422 |
| High ring counts (e.g. 38) | Unusual but genuine active day | **Keep (Valid observation)** |
| Low `days_left` (e.g. 0.1) | Depleted battery state | **Keep (Valid observation)** |


In [10]:
# Step 14 & 15: Value patterns and glitch investigations
print(f"rings_per_day: {df['rings_per_day'].nunique()} unique values")
print("Top 10 most common ring counts:")
print(df['rings_per_day'].value_counts().head(10))

print("\n--- Investigating Suspicious Readings ---")

# 1. Voltage dropouts
v_bad = df[df.battery_v < 3.0]
print(f"1. battery_v < 3.0 V: {len(v_bad)} rows (All exactly 0.0 V)")
display(v_bad)

# 2. Temperature faults
t_bad = df[df.temp_c < -10.0]
print(f"\n2. temp_c < -10.0 °C: {len(t_bad)} rows (All exactly -127.0 °C)")
display(t_bad)

# 3. Wi-Fi zero readings
w_bad = df[df.wifi_rssi_dbm >= 0]
print(f"\n3. wifi_rssi_dbm >= 0: {len(w_bad)} rows (All exactly 0 dBm)")
display(w_bad)

# Combined glitch summary
bad_mask = (df.battery_v < 3.0) | (df.temp_c < -10.0) | (df.wifi_rssi_dbm >= 0)
print("=" * 60)
print(f"Total rows in dataset      : {len(df)}")
print(f"Total glitch rows identified: {bad_mask.sum()} ({bad_mask.sum()/len(df)*100:.2f}%)")
print(f"Valid clean rows           : {(~bad_mask).sum()} ({(~bad_mask).sum()/len(df)*100:.2f}%)")
print("=" * 60)


rings_per_day: 36 unique values
Top 10 most common ring counts:
rings_per_day
4     213
3     182
6     176
5     164
2     159
7     151
8     120
9     109
10    107
1      94
Name: count, dtype: int64

--- Investigating Suspicious Readings ---
1. battery_v < 3.0 V: 9 rows (All exactly 0.0 V)


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
183,0.0,38.3,6,-86,36,29.0
313,0.0,33.4,5,-58,26,44.9
387,0.0,31.5,4,-45,21,38.4
472,0.0,42.2,5,-69,14,130.3
1185,0.0,32.2,8,-67,5,64.6
1210,0.0,28.4,9,-80,8,79.2
1458,0.0,31.9,2,-74,20,2.2
1663,0.0,33.9,2,-50,12,208.7
1773,0.0,27.4,3,-68,17,6.8



2. temp_c < -10.0 °C: 5 rows (All exactly -127.0 °C)


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
379,3.475,-127.0,32,-67,17,2.1
895,3.479,-127.0,3,-67,30,4.8
1111,3.351,-127.0,6,-45,10,0.7
1718,3.758,-127.0,3,-58,12,132.6
1863,3.499,-127.0,6,-73,15,6.3



3. wifi_rssi_dbm >= 0: 10 rows (All exactly 0 dBm)


,battery_v,temp_c,rings_per_day,wifi_rssi_dbm,age_months,days_left
83,3.374,37.3,2,0,8,0.4
94,3.786,23.7,5,0,15,75.4
594,4.009,25.9,5,0,13,61.9
913,4.025,30.7,2,0,12,75.1
928,3.762,31.5,16,0,24,52.2
1016,3.949,20.1,6,0,0,200.7
1225,3.549,26.7,18,0,3,10.3
1372,3.774,38.6,12,0,4,80.5
1433,3.448,15.5,10,0,32,2.0
1736,3.436,22.7,6,0,27,2.3


Total rows in dataset      : 2000
Total glitch rows identified: 24 (1.20%)
Valid clean rows           : 1976 (98.80%)


## Summary: What We Should NOT Do Yet

At this data inspection stage, we intentionally follow the rule:
* ❌ **Do NOT normalize the dataset yet** (normalization parameters $\mu, \sigma$ must come from the training split only!).
* ❌ **Do NOT train any neural network yet**.
* ❌ **Do NOT remove every outlier** (unusual is not invalid; keep real extremes like high ring counts or low days left).
* ❌ **Do NOT start architecture experiments yet**.

👉 **Next Step:**
Now that we have thoroughly inspected the dataset structure and identified the sensor glitches, proceed to [`eda.ipynb`](eda.ipynb) to study feature distributions (histograms), target distributions, bivariate relationship plots (`days_left` vs features), correlation heatmaps, and Mission 1 baselines!
